# COPE Program Monthly Report Automation

**Program:** COPE: Counseling, Outreach, and Protection for Emergencies
**Context:** Earthquake response psychosocial support and disability inclusion

## What This Notebook Does
1. Connects to the SQL Server database.
2. Extracts beneficiary, case action, PSS activity, and emergency cash assistance (ECA) data
   for a user-specified reporting month.
3. Codes disability categories from the `AdditionalInfo` field.
4. Deduplicates beneficiaries against a cumulative tracking file to count only *new* unique
   individuals in the current reporting period.
5. Produces five pivot tables for reporting.
6. Exports all pivot tables and raw data to a dated Excel workbook.

## Data Sources (SQL Server)
| Table | Content |
|---|---|
| `MemberTable` / `FamilyTable` | Beneficiary profiles, demographics, disability indicators |
| `CaseActionPlanTable` | Completed case action plans (caseworker-level) |
| `ActivitiesTable` / `ActivityParticipant` / `ActivityAttendance` | PSS activity attendance |
| `GerekcelendirmeFormuTable` / `ApprovedECA` | Emergency cash assistance distributions |


In [ ]:
import pandas as pd
import numpy as np
import calendar
import pyodbc
import warnings
warnings.filterwarnings("ignore")
import openpyxl
from openpyxl import load_workbook
import os
from datetime import datetime, date


## 1. Configuration

In [ ]:
# ── DATABASE ─────────────────────────────────────────────────────────────────
DB_CONNECTION_STRING = (
    "Driver={SQL Server};"
    "Server=00.0.000.00;"
    "Database=DATABASENAME;"
    "Trusted_connection=yes;"
)

# ── CASEWORKER FILTER ─────────────────────────────────────────────────────────
# Set to None to include all caseworkers in action plan extraction.
# Specify a username string to filter by the caseworkers list.
CASEWORKERS_USERNAME = "caseworkers name list"

# ── CUMULATIVE TRACKING FILE ──────────────────────────────────────────────────
CUMULATIVE_FILE = "cumulative_IDs.xlsx"

cnxn = pyodbc.connect(DB_CONNECTION_STRING)


In [ ]:
# ── REPORTING PERIOD ──────────────────────────────────────────────────────────
start_date = f'{input("Enter first day of reporting month (YYYY-MM-DD): ")} 00:00:00'
end_date   = f'{input("Enter last day of reporting month  (YYYY-MM-DD): ")} 23:59:59'

start_date = datetime.strptime(start_date, '%Y-%m-%d %H:%M:%S')
end_date   = datetime.strptime(end_date,   '%Y-%m-%d %H:%M:%S')
month_name = calendar.month_name[int(end_date.month)]

print(f"Reporting period: {start_date.date()} to {end_date.date()} ({month_name} {end_date.year})")


## 2. Helper Functions

In [ ]:
def classify_disability_code(df: pd.DataFrame, column: str) -> pd.DataFrame:
    """
    Classifies disability-related codes from the AdditionalInfo text field.
    Appends a 'disability_code' column to the DataFrame.

    Priority hierarchy (highest wins):
        ENGLBKMVRN  — Caregiver of a person with disability
        ENGLYKN     — Family member of a person with disability
        BKMVRN      — Caregiver (no disability flag on the beneficiary themselves)
        ENGL        — Person with disability
        No Code     — No disability-related code detected
    """
    categories = []
    for _, row in df[[column]].iterrows():
        val = str(row[column]).upper()
        if "ENGLBKMVRN" in val:
            categories.append("ENGLBKMVRN")
        elif "ENGLYKN" in val:
            categories.append("ENGLYKN")
        elif "BKMVRN" in val or "BKVRN" in val:
            categories.append("BKMVRN")
        elif "ENGL" in val:
            categories.append("ENGL")
        else:
            categories.append("No Code")
    df["disability_code"] = categories
    return df


## 3. Data Extraction

### 3.1 Beneficiary (Member) Data

In [ ]:
member_raw = pd.read_sql(f"""
SELECT
    MemberTable.id,
    MemberTable.UserName,
    MemberTable.RegisterDate,
    MemberTable.FamilyID,
    MemberTable.Relative,
    MemberTable.FullName,
    MemberTable.FatherName,
    MemberTable.FamilyNumber,
    MemberTable.TrID,
    MemberTable.Telephone,
    MemberTable.Gender,
    MemberTable.MaritalStatus,
    MemberTable.DateOfBirth,
    MemberTable.Nationality,
    MemberTable.AdditionalInfo,
    MemberTable.RiskLevel,
    MemberTable.CaseID,
    MemberTable.ReachedBy,
    MemberTable.ConsentForm,
    FamilyTable.RegisterDate       AS family_register_date,
    FamilyTable.ReferalFrom        AS family_referral_source,
    FamilyTable.FirstDate          AS family_first_contact,
    FamilyTable.TelephoneOfPA      AS family_phone,
    FamilyTable.Address            AS family_address,
    FamilyTable.TotalFamilyNumbersInSameHouse AS household_size,
    FamilyTable.ReachedBy          AS family_reached_by,
    FamilyTable.AdditionalInfo     AS family_additional_info,
    FamilyTable.ConsentForm        AS family_consent,
    MemberTable.UserProject,
    FLOOR(DATEDIFF(dd, MemberTable.DateOfBirth, MemberTable.RegisterDate) / 365.25) AS age,
    CASE
        WHEN FLOOR(DATEDIFF(dd, MemberTable.DateOfBirth, MemberTable.RegisterDate) / 365.25) >= 18
            THEN 'Adult'
        ELSE 'Child'
    END AS age_cohort,
    CASE
        WHEN MemberTable.Nationality = 'Syria'   THEN 'Syrian'
        WHEN MemberTable.Nationality = 'Afghan'  THEN 'Afghan'
        WHEN MemberTable.Nationality = 'Turkey'  THEN 'Turkish'
        ELSE 'Other Nationalities'
    END AS nationality_group,
    CASE
        WHEN MemberTable.AdditionalInfo LIKE '%ENGL%'    THEN 'Person with Disability'
        WHEN MemberTable.AdditionalInfo LIKE '%BKMVRN%'  THEN 'Disability Caregiver'
        WHEN MemberTable.AdditionalInfo LIKE '%ENGLYKN%' THEN 'Disability-Affected Family Member'
    END AS disability_category
FROM
    MemberTable
    RIGHT OUTER JOIN FamilyTable ON MemberTable.FamilyID = FamilyTable.id
WHERE
    MemberTable.id IS NOT NULL
    AND MemberTable.UserProject = 'COPE'
    AND MemberTable.RegisterDate >= '{start_date}'
    AND MemberTable.RegisterDate <= '{end_date}';
""", cnxn)

member_raw["AdditionalInfo"] = member_raw["AdditionalInfo"].astype(str)
member_coded = classify_disability_code(member_raw, "AdditionalInfo")
print(f"Members extracted: {len(member_raw):,}")


### 3.2 Completed Case Action Plans

In [ ]:
# Build optional caseworker filter
_caseworker_clause = (
    f"\n    AND CaseActionPlanTable.ActionCompleteUser = '{CASEWORKERS_USERNAME}'"
    if CASEWORKERS_USERNAME else ""
)

action_raw = pd.read_sql(f"""
SELECT
    MemberTable.id,
    MemberTable.RegisterDate,
    MemberTable.FullName,
    MemberTable.TrID,
    MemberTable.Gender,
    MemberTable.DateOfBirth,
    MemberTable.Nationality,
    MemberTable.Relative,
    MemberTable.FatherName,
    MemberTable.FamilyNumber,
    MemberTable.Telephone,
    MemberTable.AdditionalInfo,
    MemberTable.UserName,
    MemberTable.FamilyID,
    FLOOR(DATEDIFF(dd, MemberTable.DateOfBirth, CaseActionPlanTable.ActionCompleteDate) / 365.25) AS age,
    CASE
        WHEN FLOOR(DATEDIFF(dd, MemberTable.DateOfBirth, CaseActionPlanTable.ActionCompleteDate) / 365.25) >= 18
            THEN 'Adult'
        ELSE 'Child'
    END AS age_cohort,
    CASE
        WHEN MemberTable.Nationality = 'Syria'   THEN 'Syrian'
        WHEN MemberTable.Nationality = 'Afghan'  THEN 'Afghan'
        WHEN MemberTable.Nationality = 'Turkey'  THEN 'Turkish'
        ELSE 'Other Nationalities'
    END AS nationality_group,
    MemberTable.RiskLevel,
    CaseActionPlanTable.id                          AS case_action_id,
    CaseActionPlanTable.UserName                    AS caseworker_username,
    CaseActionPlanTable.ActionCompleteUserOffice    AS caseworker_office,
    CaseActionPlanTable.ActionCompleteUserProject   AS caseworker_project,
    CaseActionPlanTable.RegisterDate                AS action_register_date,
    CaseActionPlanTable.CaseWorker                  AS caseworker_name,
    CaseActionPlanTable.ActionPlanDate              AS action_plan_date,
    CaseActionPlanTable.SpecificNeed                AS specific_need,
    CaseActionPlanTable.Goal                        AS goal,
    CaseActionPlanTable.ReferalType                 AS referral_type,
    CaseActionPlanTable.ReferalList                 AS referral_list,
    CaseActionPlanTable.Action                      AS action,
    CaseActionPlanTable.SubAction                   AS sub_action,
    CaseActionPlanTable.ActionDetails               AS action_details,
    CaseActionPlanTable.ActionStatus                AS action_status,
    CaseActionPlanTable.ActionCompleteDate          AS action_complete_date,
    CaseActionPlanTable.ActionCompleteUser          AS action_complete_user,
    CaseActionPlanTable.GoalStatus                  AS goal_status,
    CaseActionPlanTable.ReachedBy                   AS case_reached_by
FROM
    CaseActionPlanTable
    LEFT OUTER JOIN MemberTable ON CaseActionPlanTable.MemberID = MemberTable.id
WHERE
    MemberTable.id IS NOT NULL
    AND CaseActionPlanTable.ActionCompleteUserProject = 'COPE'
    AND CaseActionPlanTable.ActionStatus = 'Completed'
    AND CaseActionPlanTable.ActionCompleteDate >= '{start_date}'
    AND CaseActionPlanTable.ActionCompleteDate <= '{end_date}'
    {{_caseworker_clause}};
""".format(_caseworker_clause=_caseworker_clause), cnxn)

print(f"Action plans extracted: {len(action_raw):,}")


### 3.3 PSS Activity Attendance

In [ ]:
activity_raw = pd.read_sql(f"""
SELECT
    ActivitiesTable.id                          AS activity_id,
    ActivitiesTable.UserName                    AS activity_username,
    ActivitiesTable.RegisterDate                AS activity_register_date,
    ActivitiesTable.StructureOfPSS              AS activity_structure,
    ActivitiesTable.ActivityType                AS activity_type,
    ActivitiesTable.SubActivitiyType            AS activity_subtype,
    ActivitiesTable.ActivityName                AS activity_name,
    ActivitiesTable.AdditionalInfo              AS activity_additional_info,
    ActivitiesTable.StartDate                   AS activity_start_date,
    ActivitiesTable.EndDate                     AS activity_end_date,
    ActivitiesTable.How_activity_was_conducted  AS delivery_mode,
    ActivitiesTable.OrganizedBy                 AS organized_by,
    ActivitiesTable.InformationDissemination    AS information_dissemination,
    ActivitiesTable.PartneredWith               AS partner_organization,
    ActivitiesTable.Location                    AS location,
    ActivityAttendance.UserOffice               AS attendance_office,
    ActivityAttendance.UserProject              AS attendance_project,
    ActivityAttendance.RegisterDate             AS attendance_register_date,
    ActivityAttendance.AttendanceDate           AS attendance_date,
    ActivityParticipant.MemberID                AS participant_member_id,
    ActivityAttendance.MemberID                 AS attendance_member_id,
    ActivityParticipant.TrID                    AS TrID,
    ActivityAttendance.TrID                     AS attendance_tr_id,
    ActivityAttendance.FullName                 AS attendance_full_name,
    ActivityAttendance.DateOfBirth              AS attendance_dob,
    ActivityAttendance.Gender                   AS Gender,
    ActivityAttendance.Nationality              AS attendance_nationality,
    ActivityParticipant.Vulnerability           AS disability_status,
    MemberTable.FamilyID,
    MemberTable.AdditionalInfo,
    ActivityAttendance.TrID + ActivityAttendance.FullName AS id_name_concat,
    IIF(
        MemberTable.TrID IN (
            SELECT TrID FROM SpecificNeedsTable WHERE Category = 'Disability'
        ), 'Yes', NULL
    ) AS disability_flag,
    IIF(MemberTable.id IS NOT NULL, 'Present', 'Absent') AS biodata_record,
    FLOOR(DATEDIFF(DAY, ActivityParticipant.DateOfBirth, ActivityAttendance.AttendanceDate) / 365.25) AS age,
    CASE
        WHEN FLOOR(DATEDIFF(DAY, ActivityParticipant.DateOfBirth, ActivityAttendance.AttendanceDate) / 365.25) >= 18
            THEN 'Adult'
        ELSE 'Child'
    END AS age_cohort,
    CASE
        WHEN FLOOR(DATEDIFF(DAY, ActivityParticipant.DateOfBirth, ActivityAttendance.AttendanceDate) / 365.25) < 6
            THEN '0-5'
        WHEN FLOOR(DATEDIFF(DAY, ActivityParticipant.DateOfBirth, ActivityAttendance.AttendanceDate) / 365.25) BETWEEN 6 AND 17
            THEN '6-17'
        ELSE '18+'
    END AS age_cohort_social_cohesion,
    CASE
        WHEN FLOOR(DATEDIFF(DAY, ActivityParticipant.DateOfBirth, ActivityAttendance.AttendanceDate) / 365.25) < 6
            THEN '0-5'
        WHEN FLOOR(DATEDIFF(DAY, ActivityParticipant.DateOfBirth, ActivityAttendance.AttendanceDate) / 365.25) BETWEEN 6 AND 13
            THEN '6-13'
        WHEN FLOOR(DATEDIFF(DAY, ActivityParticipant.DateOfBirth, ActivityAttendance.AttendanceDate) / 365.25) BETWEEN 14 AND 17
            THEN '14-17'
        ELSE '18+'
    END AS age_cohort_cefm,
    CASE
        WHEN ActivityParticipant.Nationality LIKE '%Syria%'   THEN 'Syrian'
        WHEN ActivityParticipant.Nationality LIKE '%Afghan%'  THEN 'Afghan'
        WHEN ActivityParticipant.Nationality LIKE '%Turkey%'  THEN 'Turkish'
        WHEN ActivityParticipant.Nationality LIKE '%Ukraine%' THEN 'Ukrainian'
        ELSE 'Other Nationalities'
    END AS Nationality
FROM
    ActivitiesTable
    INNER JOIN ActivityParticipant ON ActivitiesTable.id = ActivityParticipant.ActivityID
    INNER JOIN ActivityAttendance  ON ActivityParticipant.id = ActivityAttendance.ParticipantID
    LEFT  OUTER JOIN MemberTable   ON MemberTable.TrID = ActivityParticipant.TrID
WHERE
    ActivitiesTable.id IS NOT NULL
    AND ActivityAttendance.UserProject = 'COPE'
    AND ActivityAttendance.AttendanceDate BETWEEN '{start_date}' AND '{end_date}'
    AND ActivityAttendance.AttendanceStatus = 'YES';
""", cnxn)

activity_raw = classify_disability_code(activity_raw, "AdditionalInfo")
print(f"Activity attendance records extracted: {len(activity_raw):,}")


### 3.4 Emergency Cash Assistance (ECA)

In [ ]:
eca_raw = pd.read_sql(f"""
SELECT
    MemberTable.id                                          AS member_id,
    MemberTable.TrID,
    MemberTable.FullName                                    AS full_name,
    GerekcelendirmeFormuTable.YardimNedeni                  AS assistance_reason,
    GerekcelendirmeFormuTable.Belgelendirme                 AS documentation,
    GerekcelendirmeFormuTable.Hassasiyet                    AS vulnerability,
    GerekcelendirmeFormuTable.YardimTuru                    AS assistance_type,
    GerekcelendirmeFormuTable.Miktar                        AS amount,
    GerekcelendirmeFormuTable.KisiSayisi                    AS beneficiary_count,
    GerekcelendirmeFormuTable.ProjeAdi                      AS project_name,
    GerekcelendirmeFormuTable.ApproveStatus                 AS approval_status,
    GerekcelendirmeFormuTable.AdditionalInfo                AS additional_info,
    MemberTable.Gender,
    MemberTable.Nationality,
    MemberTable.DateOfBirth,
    MemberTable.RiskLevel,
    GerekcelendirmeFormuTable.UserOffice                    AS office,
    FLOOR(DATEDIFF(DAY, MemberTable.DateOfBirth, GerekcelendirmeFormuTable.DeliveryDate) / 365.25) AS age,
    CASE
        WHEN FLOOR(DATEDIFF(DAY, MemberTable.DateOfBirth, GerekcelendirmeFormuTable.DeliveryDate) / 365.25) >= 18
            THEN '+18'
        ELSE '-18'
    END AS age_cohort,
    CONVERT(DATE, GerekcelendirmeFormuTable.DeliveryDate)   AS delivery_date,
    FORMAT(GerekcelendirmeFormuTable.DeliveryDate, 'yyyy-MM') AS year_month,
    GerekcelendirmeFormuTable.*,
    ApprovedECA.*
FROM
    GerekcelendirmeFormuTable
    INNER JOIN ApprovedECA  ON GerekcelendirmeFormuTable.id = ApprovedECA.EcaID
    INNER JOIN MemberTable  ON ApprovedECA.MemberID = MemberTable.id
WHERE
    GerekcelendirmeFormuTable.ProjeAdi = 'COPE'
    AND GerekcelendirmeFormuTable.DeliveryDate BETWEEN '{start_date}' AND '{end_date}';
""", cnxn)

# The wildcard selects (GerekcelendirmeFormuTable.* and ApprovedECA.*) produce duplicate column names.
# Drop duplicates to keep only the first occurrence of each column name.
eca_raw = eca_raw.loc[:, ~eca_raw.columns.duplicated()]
eca_raw["TrID"] = eca_raw["TrID"].astype("Int64")
print(f"ECA records extracted: {len(eca_raw):,}")


## 4. Data Transformation

### 4.1 Activity Classification

In [ ]:
activity_raw["activity_type"]   = activity_raw["activity_type"].astype(str)
activity_raw["activity_subtype"] = activity_raw["activity_subtype"].astype(str)
activity_raw["reporting_month"] = f"{start_date.month}-{start_date.year}"

# Awareness Raising activities
awareness_raising = activity_raw[activity_raw["activity_type"] == "Awareness Raising"].copy()
awareness_raising["service_type"] = "Awareness Raising"

# Level 2 PSS — Recreational Activities
recreational_activities = activity_raw[
    (activity_raw["activity_type"] == "PSS") &
    (activity_raw["activity_subtype"] == "Level 2")
].copy()
recreational_activities["service_type"]     = "Recreational Activities"
recreational_activities["activity_subtype"] = "Recreational Activities"

# Level 3 PSS — Support Group Inception Sessions
support_groups = activity_raw[
    (activity_raw["activity_type"] == "PSS") &
    (activity_raw["activity_subtype"] == "Level 3")
].copy()
support_groups["service_type"]     = "Support Group Inception Sessions"
support_groups["activity_subtype"] = "Support Group Inception Sessions"


### 4.2 Deduplication

In [ ]:
# Deduplicate by TrID (individual) or FamilyID (household) within service type
activity_dedup              = activity_raw.drop_duplicates("activity_id")
awareness_raising_dedup     = awareness_raising.drop_duplicates(subset="TrID")
recreational_activities_dedup = recreational_activities.drop_duplicates(subset="TrID")
support_groups_family_dedup = support_groups.drop_duplicates(subset="FamilyID")

# Deduplicate actions by individual
action_dedup = action_raw.drop_duplicates(subset="TrID")
action_dedup["service_type"]     = "Case Action"
action_dedup["reporting_month"]  = f"{start_date.month}-{start_date.year}"
action_dedup["TrID"]             = action_dedup["TrID"].astype("Int64")


## 5. Cumulative Tracking Update

Filters out any beneficiary already recorded in `cumulative_IDs.xlsx` for each service type,
ensuring only *new* individuals are counted toward the cumulative total.

> **Note:** Beneficiaries attending multiple service types (Awareness Raising, Level 2, Level 3)
> are **not** deduplicated across service types. A beneficiary attending two service types is
> counted once per type.


In [ ]:
cumulative_ids = pd.read_excel(CUMULATIVE_FILE, sheet_name="TrID")
cumulative_ids["TrID"] = cumulative_ids["TrID"].astype("Int64")

# Retype TrID columns before anti-join filtering
awareness_raising_dedup["TrID"]         = awareness_raising_dedup["TrID"].astype("Int64")
recreational_activities_dedup["TrID"]   = recreational_activities_dedup["TrID"].astype("Int64")
support_groups_family_dedup["TrID"]     = support_groups_family_dedup["TrID"].astype("Int64")

# Anti-join: keep only TrIDs not already in cumulative file for each service type
def filter_new_beneficiaries(df: pd.DataFrame, service: str) -> pd.DataFrame:
    """Returns rows whose TrID is not already in the cumulative tracking file for this service."""
    existing = cumulative_ids[cumulative_ids["service_type"] == service]["TrID"]
    return df[~df["TrID"].isin(existing)]

action_new              = filter_new_beneficiaries(action_dedup, "Case Action")
action_new              = classify_disability_code(action_new, "AdditionalInfo")
awareness_raising_new   = filter_new_beneficiaries(awareness_raising_dedup, "Awareness Raising")
recreational_new        = filter_new_beneficiaries(recreational_activities_dedup, "Recreational Activities")
support_groups_new      = filter_new_beneficiaries(support_groups_family_dedup, "Support Group Inception Sessions")

# ECA: new direct beneficiaries only
direct_eca = eca_raw[eca_raw["DirectIndirect"] == "DIRECT"].drop_duplicates(subset="TrID")
direct_eca = filter_new_beneficiaries(direct_eca, "Case Action")
direct_eca["disability_code"]   = ""
direct_eca["service_type"]      = "ECA"
direct_eca["reporting_month"]   = f"{start_date.month}-{end_date.year}"

# Append new beneficiaries to cumulative file
cols = ["TrID", "Gender", "Nationality", "disability_code", "service_type", "reporting_month"]
cumulative_updated = pd.concat([
    cumulative_ids,
    action_new[cols],
    awareness_raising_new[cols],
    recreational_new[cols],
    support_groups_new[cols],
    direct_eca[cols],
], axis=0).reset_index(drop=True)

cumulative_updated.to_excel(CUMULATIVE_FILE, sheet_name="TrID", index=False)
print(f"Cumulative file updated: {len(cumulative_updated):,} total records")


## 6. Analytics — Pivot Tables

### 6.1 Activity Counts by Type

In [ ]:
activity_count_pivot = pd.DataFrame(
    index=["Awareness Raising", "Recreational Activities", "Support Group Inception Sessions"],
    columns=["Activity Count"]
)
activity_count_pivot.loc["Awareness Raising"]                  = len(activity_dedup[activity_dedup["activity_type"] == "Awareness Raising"])
activity_count_pivot.loc["Recreational Activities"]            = len(activity_dedup[activity_dedup["activity_subtype"] == "Level 2"])
activity_count_pivot.loc["Support Group Inception Sessions"]   = len(activity_dedup[activity_dedup["activity_subtype"] == "Level 3"])
activity_count_pivot = activity_count_pivot.fillna(0)
activity_count_pivot


### 6.2 Activity Distribution by Location

In [ ]:
support_groups_session_dedup = support_groups_new.drop_duplicates(subset="activity_id")

aw_loc = awareness_raising_dedup.drop_duplicates(subset="activity_id")
l2_loc = recreational_activities_dedup.drop_duplicates(subset="activity_id")
l3_loc = support_groups_session_dedup.drop_duplicates(subset="activity_id")

aw_loc = aw_loc.pivot_table(aw_loc, index=["location"], columns=["activity_type"],    aggfunc={"activity_id": "size"})
l2_loc = l2_loc.pivot_table(l2_loc, index=["location"], columns=["activity_subtype"], aggfunc={"activity_id": "size"})
l3_loc = l3_loc.pivot_table(l3_loc, index=["location"], columns=["activity_subtype"], aggfunc={"activity_id": "size"})

activity_by_location_pivot = pd.concat([aw_loc, l2_loc, l3_loc])
activity_by_location_pivot["Total"] = activity_by_location_pivot.sum(axis=1)
activity_by_location_pivot.loc["Total"] = activity_by_location_pivot.sum()
activity_by_location_pivot = activity_by_location_pivot.fillna(0).astype(int)
activity_by_location_pivot


### 6.3 Unique Beneficiaries by Activity Type and Demographics

In [ ]:
p1 = awareness_raising_new.pivot_table(
    awareness_raising_new, index=["activity_type"], columns=["Nationality", "Gender"],
    aggfunc={"TrID": "size"}
)
p2 = recreational_new.pivot_table(
    recreational_new, index=["activity_subtype"], columns=["Nationality", "Gender"],
    aggfunc={"TrID": "size"}
)
p3_dedup = support_groups_new.drop_duplicates(subset="FamilyID")
p3 = p3_dedup.pivot_table(
    p3_dedup, index=["activity_subtype"], columns=["Nationality", "Gender"],
    aggfunc={"TrID": "size"}
)

beneficiaries_by_activity_pivot = pd.concat([p1, p2, p3])
beneficiaries_by_activity_pivot["Total"] = beneficiaries_by_activity_pivot.sum(axis=1)
beneficiaries_by_activity_pivot.loc["Total"] = beneficiaries_by_activity_pivot.sum()
beneficiaries_by_activity_pivot = beneficiaries_by_activity_pivot.fillna(0)
beneficiaries_by_activity_pivot.rename(
    index={"Level 2": "Recreational Activities", "Level 3": "Support Group Sessions"},
    inplace=True
)
beneficiaries_by_activity_pivot = beneficiaries_by_activity_pivot.astype(int)
beneficiaries_by_activity_pivot


### 6.4 Unique Beneficiaries by Location and Demographics

In [ ]:
q1 = awareness_raising_new.pivot_table(
    awareness_raising_new, index=["location"], columns=["Nationality", "Gender"],
    aggfunc={"TrID": "size"}
)
q2 = recreational_new.pivot_table(
    recreational_new, index=["location"], columns=["Nationality", "Gender"],
    aggfunc={"TrID": "size"}
)
q3 = p3_dedup.pivot_table(
    p3_dedup, index=["location"], columns=["Nationality", "Gender"],
    aggfunc={"TrID": "size"}
)

beneficiaries_by_location_pivot = pd.concat([q1, q2, q3])
beneficiaries_by_location_pivot["Total"] = beneficiaries_by_location_pivot.sum(axis=1)
beneficiaries_by_location_pivot.loc["Total"] = beneficiaries_by_location_pivot.sum()
beneficiaries_by_location_pivot = beneficiaries_by_location_pivot.fillna(0).astype(int)
beneficiaries_by_location_pivot


### 6.5 Emergency Cash Assistance Distribution

In [ ]:
eca_summary_pivot = eca_raw.pivot_table(
    eca_raw, index=["DirectIndirect"], columns=["Nationality", "Gender"],
    aggfunc={"TrID": "size"}
)
eca_summary_pivot


## 7. Excel Export

In [ ]:
output_filename = f"COPE_Report_{end_date.year}_{month_name}.xlsx"

with pd.ExcelWriter(output_filename) as writer:
    activity_count_pivot.to_excel(writer,              sheet_name="Activity Counts")
    activity_by_location_pivot.to_excel(writer,        sheet_name="Activity by Location")
    beneficiaries_by_activity_pivot.to_excel(writer,   sheet_name="Beneficiaries by Activity")
    beneficiaries_by_location_pivot.to_excel(writer,   sheet_name="Beneficiaries by Location")
    eca_summary_pivot.to_excel(writer,                 sheet_name="ECA Summary")
    activity_raw.to_excel(writer,                      sheet_name="Activity Raw Data",  index=False)
    member_coded.to_excel(writer,                      sheet_name="Member Data",         index=False)
    eca_raw.to_excel(writer,                           sheet_name="ECA Raw Data",        index=False)

print(f"Report exported: {output_filename}")
